In [ ]:
# ============================================================
# ML WORKFLOW OVERVIEW - MEFV Mutation Pathogenicity Prediction Project
# ============================================================
# 1. Define the problem: predict whether a MEFV gene mutation is                
#    Pathogenic (harmful) or Benign (harmless), based on the mutation's
#    own properties (not patient data)
# 2. Collect data: downloaded the full MEFV variant table from INFEVERS,       
#    a curated public database for autoinflammatory disease mutations
# 3. Clean and prepare the data: simplified 8 classification categories        
#    down to 2 clear labels (Pathogenic vs Benign), dropping ambiguous
#    categories (Uncertain significance, Not classified, Unsolved, missing)
# 4. Feature engineering: turn variant properties (mutation type, protein     
#    consequence, etc.) into usable model inputs
# 5. Split the data into training and test sets                               
# 6. Choose a model                                                          
# 7. Train the model                                                          
# 8. Evaluate the model                                                       
# 9. Interpret the results                                                    
# 10. Iterate/improve                                                         
# ============================================================


## Goal: predict if a MEFV gene mutation is harmful (Pathogenic) or harmless
## (Benign), just from the mutation's own properties - not from patient data.
# Pathogenic = the mutation disrupts something important and causes disease
# Benign = the mutation is just normal, harmless human genetic variation
# (Uncertain significance / Not classified / Unsolved are excluded from this
#  first model, since they don't have a clear enough answer to train on -
#  same logic as excluding "Intermediate" in the AMR project)

In [18]:
# ============================================================
# STEP 2: Collect data - load the MEFV variant table directly from INFEVERS
# (Source: https://infevers.umai-montpellier.fr - a curated public database
#  for autoinflammatory disease mutations)
# ============================================================
import pandas as pd

url = "https://infevers.umai-montpellier.fr/upload/csv/fichier_csv.php?n=1"
df = pd.read_csv(url, encoding="latin1", sep="\t", skiprows=4)

print(df.shape)
df.head()

(420, 23)


,Location,Usual name,protein name,Sequence change,Alteration,N base(s),Base substituted,Classification,Status,Using In silico prediction?,...,Techniques used,Disease related symptoms,Associated phenotype,Zygosity in this patient,Inheritance in this patient,Ancestry/origin,Comment,Input date,References,Other PMID
0,5 flanking,-979T>C,p.?,c.-979T>C,substitution,1,T>C,Not classified,NaN,UNKNOWN,...,Sequencing Sanger,Symptomatic,FMF atypical PAAD-FMF (with criteria),Unknown,Unknown,Turkey/,This variant was identified in 6/16 controls.,2014-03-10,Akkaya-Ulum ZY Personal communication,NaN
1,5 flanking,-888G>A,p.?,c.-888G>A,substitution,1,G>A,Not classified,NaN,UNKNOWN,...,Sequencing Sanger,Symptomatic,PAAD-FMF (with criteria),Unknown,Unknown,United states/Caucasian,this variant was identified in 28/351 Caucasia...,2010-04-16,Publication (PubMed PMID): 19479870,NaN
2,5 flanking,-792C>T,p.?,c.-792C>T,substitution,1,C>T,Not classified,NaN,UNKNOWN,...,DGGE,Non symptomatic,NO,Unknown,Unknown,France/Jewish Non Ashkenasi,,2007-01-08,Notarnicola C Costa M Touitou I Persona...,NaN
3,5 flanking,-751A>G,p.?,c.-751A>G,substitution,1,A>G,Not classified,NaN,UNKNOWN,...,DGGE,Non symptomatic,NO,Unknown,Unknown,France/Maghrebian,,2007-01-08,Notarnicola C Costa M Touitou I Persona...,NaN
4,5 flanking,-740C>T,p.?,c.-740C>T,substitution,1,C>T,Not classified,NaN,UNKNOWN,...,DGGE,Non symptomatic,NO,Unknown,Unknown,Turkey/,,2007-01-08,Notarnicola C Costa M Touitou I Persona...,NaN


In [19]:
# ============================================================
# STEP 3: Clean and prepare the data
# Check what values exist in the "Classification" column (our label),
# the same way "Resistant Phenotype" was checked in the AMR project
# ============================================================
df["Classification"].value_counts(dropna=False)
# Result: 8 categories found - Uncertain significance (176), Likely benign
# (143), Likely pathogenic (36), Not classified (20), Benign (14),
# Pathogenic (12), missing (10), Unsolved (9)
# STEP 3 (continued): Simplify into 2 clear labels, drop the ambiguous ones
# ============================================================
# .map() acts like a lookup dictionary: it replaces each Classification value
# with a simpler label. Any value NOT listed here (e.g. "Uncertain
# significance", "Not classified", "Unsolved", or missing) becomes blank.
df["Label"] = df["Classification"].map({
    "Pathogenic": "Pathogenic",
    "Likely pathogenic": "Pathogenic",
    "Benign": "Benign",
    "Likely benign": "Benign"
})

# Keep only the rows that got a real label (drops the ambiguous/blank ones)
df_clean = df[df["Label"].notna()].copy()

print(df_clean.shape)
print(df_clean["Label"].value_counts())
# Result: 205 clean, usable variants - 157 Benign, 48 Pathogenic

(205, 24)
Label
Benign        157
Pathogenic     48
Name: count, dtype: int64


In [20]:
# ============================================================
# STEP 4: Feature engineering - look at what raw material is available
# to build model features from (the variant's own properties, not
# separate lookups like the AMR project needed)
# ============================================================
df_clean[["Alteration", "Sequence change", "Consequence", "Ancestry/origin"]].head(10)
# Checking these columns specifically because they likely hold the clues
# a model could learn from:
# - Alteration: the type of mutation (substitution, deletion, etc.)
# - Sequence change: the exact DNA-level change
# - Consequence: the effect on the protein (missense, nonsense, etc.) -
#   likely the most important feature, similar to how gene type mattered
#   in the AMR project
# - Ancestry/origin: which population the variant was found in

,Alteration,Sequence change,Consequence,Ancestry/origin
10,deletion,c.-46_*1114del,NaN,Turkey/Mediterranean
12,substitution,c.25C>T,NaN,Turkey/
13,substitution,c.35C>T,NaN,Turkey/
14,substitution,c.56A>G,NaN,Netherlands/
18,substitution,c.104A>G,NaN,/
20,substitution,c.124C>T,NaN,Armenia/
23,substitution,c.171G>A,NaN,Turkey/
27,substitution,c.195C>T,NaN,/
29,substitution,c.224G>A,NaN,Italy/Caucasian
30,substitution,c.250G>A,NaN,Japan/Asian


In [21]:
print(df_clean[["Alteration", "Sequence change", "Consequence", "Ancestry/origin"]].isna().sum())

Alteration           0
Sequence change      0
Consequence        194
Ancestry/origin      0
dtype: int64


In [22]:
df_clean["Alteration"].value_counts() #step4

Alteration
substitution    196
deletion          5
duplication       4
Name: count, dtype: int64

In [23]:
df_clean["Location"].value_counts() #step4

Location
Exon 2      61
Exon 10     43
Exon 3      26
Exon 5      20
Exon 1      11
Exon 8       6
Intron 5     5
Intron 8     5
3UT          5
Intron 3     4
Intron 2     4
Intron 6     3
Exon 9       3
Intron 9     3
Intron 4     2
Exon 4       2
Intron 7     1
5UT          1
Name: count, dtype: int64

In [24]:
# Simplify Location into 3 broader categories: Exon, Intron, or UTR (step 4)
df_clean["Region_Type"] = df_clean["Location"].apply(
    lambda x: "Exon" if "Exon" in str(x) else ("Intron" if "Intron" in str(x) else "UTR")
)

# Compare Region_Type between Pathogenic and Benign groups
comparison_table = pd.crosstab(df_clean["Region_Type"], df_clean["Label"], normalize="columns") * 100
print(comparison_table)

Label           Benign  Pathogenic
Region_Type                       
Exon         80.254777   95.833333
Intron       16.560510    2.083333
UTR           3.184713    2.083333


In [25]:
# Turn Region_Type into simple 0/1 columns the model can use (step 4)
features = pd.get_dummies(df_clean["Region_Type"])

# Add the label back in
model_data = features.copy()
model_data["Label"] = df_clean["Label"]

print(model_data.shape)
model_data.head()

(205, 4)


,Exon,Intron,UTR,Label
10,False,False,True,Pathogenic
12,True,False,False,Benign
13,True,False,False,Benign
14,True,False,False,Benign
18,True,False,False,Benign


In [26]:
#Step 5: Split into training and test sets
from sklearn.model_selection import train_test_split

X = model_data[["Exon", "Intron", "UTR"]]
y = model_data["Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

(164, 3) (41, 3)


In [27]:
#Step 6-7: Choose and train a model
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [28]:
#Step 8: Evaluate it
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 0.7560975609756098
              precision    recall  f1-score   support

      Benign       0.76      1.00      0.86        31
  Pathogenic       0.00      0.00      0.00        10

    accuracy                           0.76        41
   macro avg       0.38      0.50      0.43        41
weighted avg       0.57      0.76      0.65        41



/Users/alanaannaman/micromamba/envs/bioinfo/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/alanaannaman/micromamba/envs/bioinfo/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/alanaannaman/micromamba/envs/bioinfo/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [29]:
import re

# Pull out patterns like "C>T" or "A>G" from the Sequence change column
df_clean["Base_Change"] = df_clean["Sequence change"].str.extract(r'([ACGT]>[ACGT])')

print(df_clean["Base_Change"].value_counts(dropna=False))

Base_Change
G>A    49
C>T    45
A>G    22
G>C    14
T>C    13
C>G    13
C>A    10
G>T    10
NaN     9
A>T     7
T>A     7
T>G     3
A>C     3
Name: count, dtype: int64


In [30]:
base_change_features = pd.get_dummies(df_clean["Base_Change"])

model_data = pd.concat([features, base_change_features], axis=1)
model_data["Label"] = df_clean["Label"]

model_data = model_data.dropna()

print(model_data.shape)
model_data.head()

(205, 16)


,Exon,Intron,UTR,A>C,A>G,A>T,C>A,C>G,C>T,G>A,G>C,G>T,T>A,T>C,T>G,Label
10,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,Pathogenic
12,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,Benign
13,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,Benign
14,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,Benign
18,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,Benign


In [31]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = model_data.drop(columns=["Label"])
y = model_data["Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 0.8292682926829268
              precision    recall  f1-score   support

      Benign       0.82      1.00      0.90        31
  Pathogenic       1.00      0.30      0.46        10

    accuracy                           0.83        41
   macro avg       0.91      0.65      0.68        41
weighted avg       0.86      0.83      0.79        41



In [32]:
#Step 10:
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Accuracy: 0.5609756097560976
              precision    recall  f1-score   support

      Benign       0.78      0.58      0.67        31
  Pathogenic       0.28      0.50      0.36        10

    accuracy                           0.56        41
   macro avg       0.53      0.54      0.51        41
weighted avg       0.66      0.56      0.59        41



In [33]:
df_clean.to_csv("mefv_clean_data.csv", index=False)
model_data.to_csv("mefv_model_features.csv", index=False)

In [34]:
df_clean.to_csv("mefv_clean_data.csv", index=False)
model_data.to_csv("mefv_model_features.csv", index=False)

In [35]:
import os
print(os.listdir())

['mefv_model_features.csv', 'mefv.ipynb', 'mefv1.ipynb', '.ipynb_checkpoints', 'mefv_clean_data.csv']
